# 15 — Week 4: TimesFM 2.5 (200M) zero-shot on the validation split

Same protocol as notebook 14 (Chronos): same usable origins and metrics, point forecast = median (0.5 quantile),
10%/90% quantiles stored, context length chosen on validation (64, 512, 1024 minutes; TimesFM works in patches of 32).
Difference from Chronos: invalid minutes inside a context are filled by linear interpolation before the call
(leading invalid minutes are dropped), because NaN support of this package version is not verified.
Weights are read from the local cache (offline). Secondary result (leakage caveat). **No test data is touched.**
Run cell 3 first: it times 1,000 origins per context and prints the estimated full run time.

In [1]:
# Cell 1: setup
import os
os.environ["HF_HUB_OFFLINE"] = "1"
import sys, json, time, warnings, traceback
from pathlib import Path
import numpy as np
import polars as pl
import torch
import timesfm

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
from models import rnn as R
from evaluation import metrics as M
from tracking import log_run

MODEL_ID, NAME = "google/timesfm-2.5-200m-pytorch", "timesfm_2p5_200m"
CONTEXTS = [64, 512, 1024]
H_MAX = max(R.HORIZONS)
BATCH = 128
GROUP = "week4_fm"
REP = ROOT / "reports" / "week4"
try:
    model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(MODEL_ID, local_files_only=True, torch_compile=False)
except TypeError:
    model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(MODEL_ID, local_files_only=True)
n_params = int(sum(p.numel() for p in model.model.parameters())) if hasattr(model, "model") else None
dev = next(model.model.parameters()).device if hasattr(model, "model") else "?"
print(f"loaded {MODEL_ID} | params {n_params:,} | device {dev}")
COMPILED = {}


def compile_for(ctx):
    if COMPILED.get("ctx") != ctx:
        model.compile(timesfm.ForecastConfig(max_context=ctx, max_horizon=64, normalize_inputs=True,
                                             per_core_batch_size=BATCH, use_continuous_quantile_head=True,
                                             force_flip_invariance=True, infer_is_positive=True,
                                             fix_quantile_crossing=True))
        COMPILED["ctx"] = ctx


def fill(c):
    """Linear interpolation of NaN inside a context; leading NaN dropped."""
    ok = np.isfinite(c)
    if not ok.any():
        return None
    c = c[np.argmax(ok):]
    ok = np.isfinite(c)
    if not ok.all():
        k = np.arange(len(c))
        c = np.interp(k, k[ok], c[ok])
    return c.astype(np.float32)


def forecast(P, t, ctx):
    """Median, q10, q90 forecasts (N, 15) for origins t; quantile array layout checked in cell 2."""
    compile_for(ctx)
    off = np.arange(-(ctx - 1), 1)
    med, q10, q90 = [], [], []
    for i in range(0, len(t), 1024):
        idx = t[i:i + 1024, None] + off[None, :]
        raw = np.where(idx >= 0, P[np.clip(idx, 0, None)], np.nan)
        inputs = [fill(r) for r in raw]
        pt, qf = model.forecast(horizon=H_MAX, inputs=inputs)
        qf = np.asarray(qf)
        med.append(qf[:, :H_MAX, 5]); q10.append(qf[:, :H_MAX, 1]); q90.append(qf[:, :H_MAX, 9])
    return np.concatenate(med), np.concatenate(q10), np.concatenate(q90)


/opt/miniconda3/envs/dcpower/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded google/timesfm-2.5-200m-pytorch | params 231,289,280 | device cpu


In [2]:
# Cell 2: SYNTHETIC check: output layout (mean + deciles), quantile order, NaN filling, sensible accuracy
rng = np.random.default_rng(0)
n = 5000
tt = np.arange(n)
Psyn = (30 + 3 * np.sin(2 * np.pi * tt / 240) + rng.normal(0, 0.2, n)).astype(np.float32)
Psyn[1000:1100] = np.nan
orig = np.arange(1200, 4900, 37)
compile_for(512)
off = np.arange(-511, 1)
pt, qf = model.forecast(horizon=H_MAX, inputs=[fill(Psyn[o + off]) for o in orig[:8]])
pt, qf = np.asarray(pt), np.asarray(qf)
print("point shape", pt.shape, "| quantile shape", qf.shape)
print("deciles ordered:", bool((np.diff(qf[:, :H_MAX, 1:], axis=2) >= -1e-5).all()),
      "| max |point - q50|:", float(np.abs(pt[:, :H_MAX] - qf[:, :H_MAX, 5]).max()))
syn = {}
for ctx in (64, 512):
    med, q10, q90 = forecast(Psyn, orig, ctx)
    row = {"ctx": ctx, "finite": bool(np.isfinite(med).all())}
    for h in R.HORIZONS:
        y = Psyn[orig + h]
        row[f"skill_h{h}"] = round(float(1 - np.mean(np.abs(med[:, h - 1] - y)) / np.mean(np.abs(Psyn[orig] - y))), 3)
    row["cover80"] = round(float(np.mean((Psyn[orig + 15] >= q10[:, 14]) & (Psyn[orig + 15] <= q90[:, 14]))), 3)
    syn[ctx] = row
    print(row)
good = qf.shape[-1] == 10 and all(r["finite"] and r["skill_h15"] > 0 for r in syn.values())
print("\nSYNTHETIC CHECK PASSED" if good else "\nSYNTHETIC CHECK NOT PASSED: send this output")


point shape (8, 15) | quantile shape (8, 15, 10)
deciles ordered: True | max |point - q50|: 0.0
{'ctx': 64, 'finite': True, 'skill_h1': 0.146, 'skill_h5': 0.016, 'skill_h15': 0.175, 'cover80': 0.5}
{'ctx': 512, 'finite': True, 'skill_h1': 0.273, 'skill_h5': 0.454, 'skill_h15': 0.804, 'cover80': 0.91}

SYNTHETIC CHECK PASSED


In [3]:
# Cell 3: REAL data: timing on 1,000 validation origins per context (decide before the full run)
md = pl.read_parquet(ROOT / "data" / "processed" / "model_dataset.parquet")
meta = json.loads((ROOT / "data" / "processed" / "model_dataset_meta.json").read_text())
data = R.Data(md, {"scaling": "local", "calendar": False, "covariates": False})
valid = md["valid"].to_numpy()
Pnan = np.where(valid, md["P_kW"].fill_null(np.nan).to_numpy(), np.nan).astype(np.float32)
L24 = md["L24_kW"].to_numpy().astype(float)
mod5 = md["mod5"].to_numpy().astype(int)
ep = md["minute_epoch"].to_numpy()
va = data.origins("val")
sub = np.sort(np.random.default_rng(0).choice(va, 1000, replace=False))
EST = {}
for ctx in CONTEXTS:
    forecast(Pnan, sub[:32], ctx)                        # compile + warm-up
    t0 = time.time()
    med, _, _ = forecast(Pnan, sub, ctx)
    s = time.time() - t0
    EST[ctx] = s / len(sub) * len(va) / 60
    sk = []
    for h in R.HORIZONS:
        sel = np.isin(sub, data.idx[h]["val"])
        y, p0 = data.P[sub[sel] + h], data.P[sub[sel]]
        sk.append(1 - np.mean(np.abs(med[sel, h - 1] - y)) / np.mean(np.abs(p0 - y)))
    print(f"ctx {ctx:>5}: {1000 * s / len(sub):.1f} ms/origin | est. full validation {EST[ctx]:.0f} min | "
          f"skill on 1,000 origins " + " / ".join(f"{v:+.3f}" for v in sk))
print(f"\nestimated total for cell 4: {sum(EST.values()):.0f} min (remove a context from CONTEXTS if too long)")


ctx    64: 3.8 ms/origin | est. full validation 3 min | skill on 1,000 origins +0.072 / +0.050 / +0.009
ctx   512: 31.7 ms/origin | est. full validation 23 min | skill on 1,000 origins +0.108 / +0.052 / +0.056
ctx  1024: 63.8 ms/origin | est. full validation 45 min | skill on 1,000 origins +0.125 / +0.047 / +0.049

estimated total for cell 4: 70 min (remove a context from CONTEXTS if too long)


In [4]:
CONTEXTS = [64, 512]

In [5]:
# Cell 4: full validation run per context (resumable; long: see cell 3)
results = {}
for ctx in CONTEXTS:
    run = f"{NAME}_ctx{ctx}"
    out = ROOT / "experiments" / GROUP / run
    if (out / "metrics.json").exists():
        results[ctx] = json.loads((out / "metrics.json").read_text())
        print(f"{run}: done earlier, skipped")
        continue
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    MED, Q10, Q90 = forecast(Pnan, va, ctx)
    infer_s = time.time() - t0
    rows, metrics, parts = [], {}, []
    for h in R.HORIZONS:
        t = data.idx[h]["val"]
        pos = np.searchsorted(va, t)
        yhat, q10, q90 = (a[pos, h - 1].astype(float) for a in (MED, Q10, Q90))
        y, p0 = data.P[t + h], data.P[t]
        ramp = md[f"ramp_h{h}"].to_numpy().astype(float)[t]
        r = M.evaluate(y, yhat, p0, l24=L24[t], ramp=ramp, thr=meta["thresholds"][str(h)]["rel_p95"],
                       mod5_target=mod5[t + h] if h == 1 else None, dm_lag=60)
        r["cover_q10_q90"] = float(np.mean((y >= q10) & (y <= q90)))
        r["width_q10_q90"] = float(np.mean(q90 - q10))
        rows.append({"h": h, **r})
        metrics.update({f"h{h}/{k}": v for k, v in r.items() if isinstance(v, (int, float))})
        parts.append(pl.DataFrame({"minute_epoch": ep[t], "h": np.full(len(t), h, np.int16), "y": y, "p0": p0,
                                   "yhat": yhat, "q10": q10, "q90": q90}))
    pl.concat(parts).write_parquet(out / "predictions_val.parquet")
    info = {"model": MODEL_ID, "context": ctx, "n_params": n_params, "train_s": 0.0, "device": str(dev),
            "infer_s_val": infer_s, "infer_ms_per_sample": 1000 * infer_s / len(va), "nan_handling": "linear interpolation"}
    metrics.update({k: v for k, v in info.items() if isinstance(v, (int, float))})
    res = {"info": info, "val": rows, "metrics": metrics}
    log_run(ROOT, run, GROUP, {"family": "foundation_zero_shot", **info}, metrics, tables={"val_metrics": pl.DataFrame(rows)})
    (out / "metrics.json").write_text(json.dumps(res, indent=1, default=str))
    results[ctx] = res
    print(f"{run}: {infer_s / 60:.1f} min | val MAE " + " / ".join(f"{r_['MAE']:.4f}" for r_ in rows) +
          " | skill " + " / ".join(f"{r_['skill_MAE']:+.3f}" for r_ in rows) +
          " | 80% coverage " + " / ".join(f"{r_['cover_q10_q90']:.3f}" for r_ in rows))


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/aryaneuse/.netrc.


timesfm_2p5_200m_ctx64: 2.8 min | val MAE 0.2694 / 0.4133 / 0.5974 | skill +0.060 / +0.039 / +0.012 | 80% coverage 0.817 / 0.770 / 0.699
timesfm_2p5_200m_ctx512: 23.9 min | val MAE 0.2602 / 0.3976 / 0.5775 | skill +0.092 / +0.075 / +0.045 | 80% coverage 0.836 / 0.819 / 0.782


In [6]:
# Cell 5: choose the context on validation and add TimesFM to the benchmark table (next to Chronos)
summ = pl.DataFrame([{"model": f"{NAME}_ctx{c}", "context": c, "h": r_["h"], "MAE": r_["MAE"], "RMSE": r_["RMSE"],
                      "sMAPE": r_["sMAPE"], "R2_delta": r_["R2_delta"], "skill_MAE": r_["skill_MAE"],
                      "MAE_ramp": r_["MAE_ramp"], "MAE_normal": r_["MAE_normal"], "ramp_F1": r_["ramp_F1"],
                      "DM_p": r_["DM_p"], "cover_q10_q90": r_["cover_q10_q90"], "width_q10_q90": r_["width_q10_q90"],
                      "infer_ms_per_sample": res["info"]["infer_ms_per_sample"]}
                     for c, res in results.items() for r_ in res["val"]])
old = pl.read_csv(REP / "fm_val_summary.csv")
pl.concat([old.filter(~pl.col("model").str.starts_with(NAME)), summ], how="diagonal_relaxed").write_csv(REP / "fm_val_summary.csv")
pl.Config.set_tbl_rows(30); pl.Config.set_tbl_cols(16); pl.Config.set_tbl_width_chars(220)
print(summ.select("h", "context", pl.col("MAE").round(4), pl.col("skill_MAE").round(3), pl.col("MAE_ramp").round(3),
                  pl.col("MAE_normal").round(4), pl.col("ramp_F1").round(3), pl.col("cover_q10_q90").round(3),
                  pl.col("width_q10_q90").round(3), pl.col("infer_ms_per_sample").round(2)).sort("h", "context"))
best_c = summ.group_by("context").agg(pl.col("MAE").mean()).sort("MAE")["context"][0]
print(f"\ncontext chosen on validation (mean MAE over h): {best_c} min")
bench = pl.read_csv(REP / "benchmark_val_week4_all.csv").filter(pl.col("model") != NAME)
fm = (summ.filter(pl.col("context") == best_c)
          .with_columns(pl.lit("foundation").alias("family"), pl.lit(NAME).alias("model"), pl.lit(False).alias("covariates"),
                        pl.lit(1).alias("n_seeds"), pl.col("MAE").alias("MAE_mean"), pl.lit(0.0).alias("MAE_sd"))
          .select(bench.columns))
allb = pl.concat([bench, fm.with_columns([pl.col(c).cast(bench.schema[c]) for c in bench.columns])]).sort("h", "MAE_mean")
allb.write_csv(REP / "benchmark_val_week4_all.csv")
for h in R.HORIZONS:
    t = allb.filter(pl.col("h") == h)
    ms = t["model"].to_list()
    print(f"h={h:>2}: {NAME} rank {ms.index(NAME) + 1} of {t.height} | chronos_bolt_small rank "
          f"{ms.index('chronos_bolt_small') + 1} | best {ms[0]}")


shape: (6, 10)
┌─────┬─────────┬────────┬───────────┬──────────┬────────────┬─────────┬───────────────┬───────────────┬─────────────────────┐
│ h   ┆ context ┆ MAE    ┆ skill_MAE ┆ MAE_ramp ┆ MAE_normal ┆ ramp_F1 ┆ cover_q10_q90 ┆ width_q10_q90 ┆ infer_ms_per_sample │
│ --- ┆ ---     ┆ ---    ┆ ---       ┆ ---      ┆ ---        ┆ ---     ┆ ---           ┆ ---           ┆ ---                 │
│ i64 ┆ i64     ┆ f64    ┆ f64       ┆ f64      ┆ f64        ┆ f64     ┆ f64           ┆ f64           ┆ f64                 │
╞═════╪═════════╪════════╪═══════════╪══════════╪════════════╪═════════╪═══════════════╪═══════════════╪═════════════════════╡
│ 1   ┆ 64      ┆ 0.2694 ┆ 0.06      ┆ 1.401    ┆ 0.2178     ┆ 0.261   ┆ 0.817         ┆ 0.845         ┆ 3.96                │
│ 1   ┆ 512     ┆ 0.2602 ┆ 0.092     ┆ 1.388    ┆ 0.2089     ┆ 0.211   ┆ 0.836         ┆ 0.845         ┆ 33.69               │
│ 5   ┆ 64      ┆ 0.4133 ┆ 0.039     ┆ 2.254    ┆ 0.3017     ┆ 0.29    ┆ 0.77          ┆ 1.103  

In [8]:
# Cell 6: export outputs (save the notebook first with Cmd+S)
%run ../src/tools/nb_outputs_to_md.py 15_week4_timesfm_val.ipynb


written 15_week4_timesfm_val_outputs.md | code cells 8 | figures 0
